In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly

from Strategies.LeadLagRegression_strategy.backtest_class import BacktestLL
from Strategies.LeadLagRegression_strategy.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

In [2]:
data_lead = pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lead_production_sample.csv',
                    parse_dates=['datetime']).reset_index()

data_lag = pd.read_csv(r's:\Algo\Files\andrej\Data\int_data_lag_production_sample.csv',
                    parse_dates=['datetime']).reset_index()

In [3]:
data_lead['datetime'] = pd.to_datetime(data_lead['datetime'], errors='coerce')
data_lag['datetime'] = pd.to_datetime(data_lag['datetime'], errors='coerce')

In [4]:
data_lag['time_diff']=data_lag['datetime'].diff().dt.total_seconds().fillna(0)

In [5]:
data_lead=data_lead[data_lead['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]
data_lag=data_lag[data_lag['datetime'].apply(lambda x: x.hour>8 and  x.hour<18 and x.day==22 and x.month==11)]
data_lag=data_lag[data_lag['trd_price'].isnull()==False]

In [6]:
data_lag[0:50]

,index,datetime,trd_price,volume,bid_price,ask_price,mid_price,trd_side,time_diff
123437,123437,2024-11-22 09:01:06.433828115,124.80,1.0,124.80,124.89,124.845,-1.0,0.433828
123438,123438,2024-11-22 09:01:06.434665203,124.80,1.0,124.80,124.89,124.845,-1.0,0.000837
123439,123439,2024-11-22 09:01:29.289333344,124.80,1.0,124.80,124.89,124.845,-1.0,22.854668
123441,123441,2024-11-22 09:01:47.806319475,124.80,1.0,124.80,124.88,124.840,-1.0,2.806319
123442,123442,2024-11-22 09:02:00.136390448,124.80,1.0,124.80,124.88,124.840,-1.0,12.330071
123444,123444,2024-11-22 09:02:03.148713589,124.73,1.0,124.74,124.87,124.805,-1.0,2.148714
123446,123446,2024-11-22 09:02:07.564892054,124.73,1.0,124.73,124.87,124.800,-1.0,3.564892
123449,123449,2024-11-22 09:02:10.597795725,124.73,1.0,124.73,124.86,124.795,-1.0,1.597796
123453,123453,2024-11-22 09:02:25.459381580,124.65,1.0,124.65,124.85,124.750,-1.0,10.459382
123461,123461,2024-11-22 09:03:25.885635138,124.66,1.0,124.50,124.74,124.620,1.0,5.885635


In [7]:
class EMA:
    def __init__(self, span):
        self.span = span
        self.value = 0
        self.alpha = 2 / (span + 1)

    def push(self, value):
        self.value = self.alpha * value + (1 - self.alpha) * self.value

class TR_class:
    def __init__(self, tau, tau_ema, burn=10):
        self.tau = tau
        self.tau_ema = tau_ema
        self.reset()
        self.__burn = burn

    @property
    def param_keys(self):
        return ['tau', 'tau_ema']

    def update_params(self, params_dict):
        if 'tau' in params_dict.keys():
            self.tau = params_dict['tau']
        if 'tau_ema' in params_dict.keys():
            self.tau_ema = params_dict['tau_ema']

    @property
    def ewma_val(self):
        return self.ewma.value

    @property
    def is_burn(self):
        return self.tot_n < self.burn

    @property
    def min_tau(self):
        return self.tau // 3

    @property
    def old_value(self):
        return self.__old_value

    @property
    def bt(self):
        return self.__bt

    @property
    def burn(self):
        return self.__burn

    @property
    def tot_n(self):
        return self.__tot_n

    def reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index = 0
        self.__bt = 1
        self.__old_value = np.nan
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def soft_reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index += 1
        self.__bt = 1
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def initialize(self):
        self.ewma.value = .5
        self.ewma_T.value = self.tau
        self.init = False

    def push(self, value, volume=None):
        if self.tot_n < 1:
            self.init = True
        else:
            diff_value = value - self.old_value
            self.__bt = self.signed_tick_vals(diff_value)
            bt = max(self.__bt, 0)
            if self.init:
                self.initialize()
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
            if self.is_burn:
                self.phiT += bt
                self.__n += 1
            elif max(self.phiT, self.__n - self.phiT) < self.thres:
                self.phiT += bt
                self.__n += 1
            else:
                self.ewma.push(self.phiT / self.__n)
                self.ewma_T.push(self.__n)
                self.phiT = 0
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
                self.index += 1
                self.__n = 0
        self.__tot_n += 1
        self.__old_value = value
        return self.index

    def signed_tick_vals(self, diff_value):
        if diff_value > 0:
            return 1
        elif diff_value < 0:
            return -1
        else:
            return 0

    def tick_imbalance_single(self, trades):
        self.soft_reset()
        index_series = []
        for trade in trades:
            value = trade[0]
            if not value or np.isnan(value):
                index_series.append((trade[2], self.index))
            else:
                index_series.append((trade[2], self.push(value)))

        return index_series

    def tick_imbalance_indices(self, trades):
        self.reset()
        index_series = []
        current_date = None
        daily_trades = []

        for trade in trades:
            trade_date = trade[2].date()
            if current_date is None:
                current_date = trade_date

            if trade_date != current_date:
                # Process the previous day's trades
                index_series.extend(self.tick_imbalance_single(daily_trades))
                daily_trades = []
                current_date = trade_date

            daily_trades.append(trade)

        # Process the last day's trades
        if daily_trades:
            index_series.extend(self.tick_imbalance_single(daily_trades))

        return index_series

In [8]:
# Preparing tick data
ti_cls = TR_class(tau=10, tau_ema=10)

df_list = []
df_list2 = []

df_trds_1day=data_lag
df_trds_1day['execution_time'] = df_trds_1day['datetime'].astype('int64')  # Already in nanoseconds

sort_order=[True, False, True]

df_trds_1day=df_trds_1day.reset_index()
df_trds_1day = df_trds_1day.sort_values(by=['datetime', 'volume', 'trd_price'], ascending=sort_order)


# Create the list of tuples
lag_trades = [(row['trd_price'], row['volume'], row['execution_time']) for index, row in df_trds_1day.iterrows()]

idx_series = ti_cls.tick_imbalance_single(lag_trades)
idx_series = pd.DataFrame(idx_series, columns=['index', 0])

if 'level_0' in df_trds_1day.columns:
    del df_trds_1day['level_0']

# Create returns
df_trds_indexed_1day = pd.concat([df_trds_1day, idx_series[0]], axis=1).reset_index()


df_trds_indexed_1day['tick_id'] = df_trds_indexed_1day[0].fillna(0).apply(str)

In [9]:
df_trds_indexed_1day[0:51]

,level_0,index,datetime,trd_price,volume,bid_price,ask_price,mid_price,trd_side,time_diff,execution_time,0,tick_id
0,0,123437,2024-11-22 09:01:06.433828115,124.80,1.0,124.80,124.89,124.845,-1.0,0.433828,1732266066433828115,1,1
1,1,123438,2024-11-22 09:01:06.434665203,124.80,1.0,124.80,124.89,124.845,-1.0,0.000837,1732266066434665203,1,1
2,2,123439,2024-11-22 09:01:29.289333344,124.80,1.0,124.80,124.89,124.845,-1.0,22.854668,1732266089289333344,1,1
3,3,123441,2024-11-22 09:01:47.806319475,124.80,1.0,124.80,124.88,124.840,-1.0,2.806319,1732266107806319475,1,1
4,4,123442,2024-11-22 09:02:00.136390448,124.80,1.0,124.80,124.88,124.840,-1.0,12.330071,1732266120136390448,1,1
5,5,123444,2024-11-22 09:02:03.148713589,124.73,1.0,124.74,124.87,124.805,-1.0,2.148714,1732266123148713589,1,1
6,6,123446,2024-11-22 09:02:07.564892054,124.73,1.0,124.73,124.87,124.800,-1.0,3.564892,1732266127564892054,1,1
7,7,123449,2024-11-22 09:02:10.597795725,124.73,1.0,124.73,124.86,124.795,-1.0,1.597796,1732266130597795725,1,1
8,8,123453,2024-11-22 09:02:25.459381580,124.65,1.0,124.65,124.85,124.750,-1.0,10.459382,1732266145459381580,1,1
9,9,123461,2024-11-22 09:03:25.885635138,124.66,1.0,124.50,124.74,124.620,1.0,5.885635,1732266205885635138,1,1


In [10]:
import numpy as np
from datetime import datetime, time
import pytz
import logging
import math

log = logging.getLogger("leadlag_strategy.predictors")


######################################################################################
### helper functions
class EMA:
    def __init__(self, span):
        self.span = span
        self.value = 0
        self.alpha = 2 / (span + 1)

    def push(self, value):
        self.value = self.alpha * value + (1 - self.alpha) * self.value



class TR_class:
    def __init__(self, tau, tau_ema, burn=10):
        self.tau = tau
        self.tau_ema = tau_ema
        self.reset()
        self.__burn = burn

    @property
    def param_keys(self):
        return ['tau', 'tau_ema']

    def update_params(self, params_dict):
        if 'tau' in params_dict.keys():
            self.tau = params_dict['tau']
        if 'tau_ema' in params_dict.keys():
            self.tau_ema = params_dict['tau_ema']

    @property
    def ewma_val(self):
        return self.ewma.value

    @property
    def is_burn(self):
        return self.tot_n < self.burn

    @property
    def min_tau(self):
        return self.tau // 3

    @property
    def old_value(self):
        return self.__old_value

    @property
    def bt(self):
        return self.__bt

    @property
    def burn(self):
        return self.__burn

    @property
    def tot_n(self):
        return self.__tot_n

    def reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index = 0
        self.__bt = 1
        self.__old_value = np.nan
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def soft_reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index += 1
        self.__bt = 1
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def initialize(self):
        self.ewma.value = .5
        self.ewma_T.value = self.tau
        self.init = False

    def push(self, value, volume=None):
        if self.tot_n < 1:
            self.init = True
        else:
            diff_value = value - self.old_value
            self.__bt = self.signed_tick_vals(diff_value)
            bt = max(self.__bt, 0)
            if self.init:
                self.initialize()
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
            if self.is_burn:
                self.phiT += bt
                self.__n += 1
            elif max(self.phiT, self.__n - self.phiT) < self.thres:
                self.phiT += bt
                self.__n += 1
            else:
                self.ewma.push(self.phiT / self.__n)
                self.ewma_T.push(self.__n)
                self.phiT = 0
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
                self.index += 1
                self.__n = 0
        self.__tot_n += 1
        self.__old_value = value
        return None if self.is_burn else self.index

    def signed_tick_vals(self, diff_value):
        if diff_value > 0:
            return 1
        elif diff_value < 0:
            return -1
        else:
            return 0

    def tick_imbalance_single(self, trades):
        self.soft_reset()
        index_series = []
        for trade in trades:
            value = trade[0]
            index_series.append((trade[0], trade[1], trade[2], self.push(value)))
        return index_series

    def tick_imbalance_indices(self, trades):
        self.reset()
        index_series = []
        current_date = None
        daily_trades = []

        for trade in trades:
            trade_date = trade[2].date()
            if current_date is None:
                current_date = trade_date

            if trade_date != current_date:
                # Process the previous day's trades
                index_series.extend(self.tick_imbalance_single(daily_trades))
                daily_trades = []
                current_date = trade_date

            daily_trades.append(trade)

        # Process the last day's trades
        if daily_trades:
            index_series.extend(self.tick_imbalance_single(daily_trades))

        return index_series



def get_nine_am_unix_today_cet():
    # Define the CET timezone
    cet = pytz.timezone('CET')

    # Get today's date in the CET timezone
    today = datetime.now(cet).date()

    # Combine today's date with the time 09:00 AM in CET
    nine_am_today = cet.localize(datetime.combine(today, time(9, 0)))

    # Convert to Unix timestamp (seconds since epoch)
    unix_timestamp = int(nine_am_today.timestamp())

    return unix_timestamp


def calculate_ema(current_price, previous_ema, span):
    alpha = 2 / (span + 1)
    return alpha * current_price + (1 - alpha) * previous_ema

######################################################################################
### predictor calculation

In [11]:
lag_trades=[(124.8, 1.0, 1732262466.433828), (124.8, 1.0, 1732262466.4346652), (124.8, 1.0, 1732262489.289333), (124.8, 1.0, 1732262507.8063192), (124.8, 1.0, 1732262520.1363904), (124.73, 1.0, 1732262523.1487136), (124.73, 1.0, 1732262527.564892), (124.73, 1.0, 1732262530.5977955), (124.65, 1.0, 1732262545.4593816), (124.66, 1.0, 1732262605.8856351), (124.2, 2.0, 1732262631.3390172), (124.2, 3.0, 1732262639.6083572), (124.19, 2.0, 1732262639.6083572), (124.09, 3.0, 1732262707.484876), (124.09, 1.0, 1732262707.484876), (124.09, 1.0, 1732262707.485064), (124.07, 1.0, 1732262719.679844), (123.84, 1.0, 1732262842.299234), (123.96, 1.0, 1732262844.7605236), (123.96, 1.0, 1732262844.8716373), (123.96, 1.0, 1732262845.0919175), (123.96, 1.0, 1732262845.0919445), (123.96, 1.0, 1732262845.2807796), (123.96, 1.0, 1732262845.2808151), (123.96, 1.0, 1732262845.4916384), (123.6, 1.0, 1732262909.4625115), (123.56, 1.0, 1732262909.674972), (123.52, 1.0, 1732262909.716689), (123.69, 1.0, 1732262909.928343), (123.59, 1.0, 1732262911.0334103), (123.49, 2.0, 1732262917.2673912), (123.4, 1.0, 1732262917.2674224), (123.48, 1.0, 1732262919.058424), (123.47, 1.0, 1732262919.0584483), (123.57, 1.0, 1732262919.192144), (123.58, 1.0, 1732262921.976752), (123.58, 1.0, 1732262921.9767828), (123.58, 1.0, 1732262922.0873177), (123.43, 1.0, 1732263029.5342097), (123.43, 1.0, 1732263029.5352263), (123.5, 1.0, 1732263032.101565), (123.63, 1.0, 1732263092.826254), (123.64, 1.0, 1732263093.1513429), (124.04, 1.0, 1732263225.0483048), (123.92, 1.0, 1732263230.8447587), (123.92, 1.0, 1732263230.846377), (123.92, 1.0, 1732263230.8465445), (123.86, 1.0, 1732263345.2535613), (123.86, 1.0, 1732263345.6330285), (123.86, 2.0, 1732263395.9139512), (123.86, 1.0, 1732263400.873726), (123.86, 1.0, 1732263401.1566088), (123.86, 1.0, 1732263408.8490832), (123.86, 1.0, 1732263408.8491116), (123.86, 1.0, 1732263408.9564435), (123.86, 1.0, 1732263408.9564593), (123.86, 1.0, 1732263411.8198357), (123.86, 1.0, 1732263411.8198614), (123.86, 1.0, 1732263412.62864), (123.86, 1.0, 1732263415.169446), (123.86, 1.0, 1732263415.4301343), (123.86, 1.0, 1732263415.4337177), (123.86, 2.0, 1732263419.6224597), (123.86, 2.0, 1732263420.0251353), (123.86, 2.0, 1732263421.166777), (123.86, 2.0, 1732263421.2972033), (123.86, 1.0, 1732263421.4431121), (123.65, 1.0, 1732263515.4824772), (123.66, 1.0, 1732263515.5056925), (123.64, 5.0, 1732263551.0983095), (123.42, 2.0, 1732263605.4053323), (122.78, 2.0, 1732263717.317354), (122.78, 2.0, 1732263720.3209867), (122.78, 2.0, 1732263722.3837113), (122.78, 2.0, 1732263723.8532429), (122.78, 2.0, 1732263723.928151), (122.78, 2.0, 1732263724.2447717), (122.78, 2.0, 1732263724.2569103), (122.78, 2.0, 1732263726.4027317), (122.78, 2.0, 1732263726.5359356), (122.78, 2.0, 1732263726.722818), (122.78, 2.0, 1732263727.2647045), (122.78, 2.0, 1732263727.2981796), (122.78, 1.0, 1732263727.3568206), (122.75, 1.0, 1732263777.6723135), (122.79, 2.0, 1732263971.049451), (122.6, 1.0, 1732264024.141272), (122.6, 1.0, 1732264024.1413188), (122.6, 1.0, 1732264024.2426136), (122.77, 1.0, 1732264033.628276), (122.7, 1.0, 1732264046.2683504), (122.78, 2.0, 1732264055.0947435), (122.78, 1.0, 1732264055.0948143), (122.78, 1.0, 1732264058.3416924), (122.78, 1.0, 1732264062.5171173), (122.8, 1.0, 1732264065.168232), (122.8, 1.0, 1732264065.1682572), (122.9, 1.0, 1732264067.7119935), (122.79, 1.0, 1732264071.8337147), (122.78, 1.0, 1732264072.174244), (122.65, 1.0, 1732264177.5414178), (122.65, 4.0, 1732264177.541572), (122.74, 2.0, 1732264200.5370708), (122.74, 1.0, 1732264203.5882275), (122.59, 2.0, 1732264203.685666), (122.7, 1.0, 1732264211.2231877), (122.98, 1.0, 1732264277.9315712), (123.0, 1.0, 1732264287.1315215), (123.1, 1.0, 1732264294.9049022), (123.05, 1.0, 1732264334.553978), (123.05, 1.0, 1732264334.5622401), (123.05, 1.0, 1732264334.6509783), (123.12, 2.0, 1732264402.0671556), (123.11, 1.0, 1732264405.533507), (123.12, 1.0, 1732264405.7445977), (123.15, 1.0, 1732264413.221269), (123.13, 1.0, 1732264422.214013), (123.17, 1.0, 1732264433.8296237), (123.2, 1.0, 1732264440.6030767), (123.28, 2.0, 1732264475.9201791), (123.0, 1.0, 1732264640.7075405), (122.97, 1.0, 1732264647.9184515), (122.97, 1.0, 1732264648.9393616), (122.97, 1.0, 1732264648.953448), (122.97, 1.0, 1732264649.156391), (122.97, 1.0, 1732264652.365188), (122.97, 1.0, 1732264652.6575365), (122.89, 2.0, 1732264690.4212916), (122.89, 3.0, 1732264690.4214046), (122.87, 1.0, 1732264697.5362103), (122.82, 1.0, 1732264703.3984687), (122.77, 1.0, 1732264736.7164917), (122.76, 1.0, 1732264736.7303934), (122.76, 1.0, 1732264743.350232), (122.76, 1.0, 1732264743.362807), (122.76, 1.0, 1732264748.596666), (122.86, 1.0, 1732264961.2984667), (122.85, 1.0, 1732264971.247394), (122.81, 1.0, 1732264982.628306), (122.8, 4.0, 1732264982.628413), (122.8, 1.0, 1732265017.8152835), (122.8, 1.0, 1732265017.8153527), (122.8, 1.0, 1732265017.8154016), (122.8, 1.0, 1732265017.8154511), (122.79, 1.0, 1732265017.816246), (122.7, 1.0, 1732265035.1770215), (122.7, 1.0, 1732265035.27337), (122.64, 1.0, 1732265189.6943004), (122.6, 1.0, 1732265203.8721757), (122.65, 1.0, 1732265257.3955245), (122.65, 1.0, 1732265257.4342976), (122.65, 1.0, 1732265257.4459908), (122.58, 1.0, 1732265261.106542), (122.57, 3.0, 1732265280.329239), (122.54, 1.0, 1732265280.329239), (122.55, 1.0, 1732265280.329239), (122.54, 1.0, 1732265280.3434408), (122.78, 1.0, 1732265313.533506), (122.6, 1.0, 1732265423.8430424), (122.56, 1.0, 1732265423.9219103), (122.56, 1.0, 1732265424.1223087), (122.55, 1.0, 1732265424.1223764), (122.43, 1.0, 1732265444.548178), (122.44, 1.0, 1732265444.548178), (122.48, 1.0, 1732265444.548178), (122.54, 1.0, 1732265444.548178), (122.69, 1.0, 1732265474.4766805), (122.69, 1.0, 1732265578.4373803), (122.7, 1.0, 1732265581.4530375), (122.7, 1.0, 1732265592.3109157), (122.94, 1.0, 1732265640.2926912), (122.95, 1.0, 1732265643.8172092), (122.95, 1.0, 1732265707.5115306), (122.95, 1.0, 1732265707.512848), (122.99, 1.0, 1732265707.5128946), (122.94, 1.0, 1732265716.7084005), (122.94, 1.0, 1732265723.4416964), (123.0, 2.0, 1732265723.5208623), (122.94, 1.0, 1732265724.4845436), (122.94, 1.0, 1732265724.8861375), (122.88, 1.0, 1732265725.2062767), (122.87, 5.0, 1732265725.2205172), (122.95, 1.0, 1732265757.9928813), (122.94, 1.0, 1732265761.629735), (122.95, 1.0, 1732265763.5700378), (122.78, 1.0, 1732265768.1986673), (122.78, 1.0, 1732265768.1988764), (122.78, 1.0, 1732265768.564126), (122.74, 1.0, 1732265768.585174), (122.74, 1.0, 1732265768.704278), (122.74, 1.0, 1732265768.7789555), (122.77, 1.0, 1732265768.779071), (122.74, 1.0, 1732265768.88221), (122.77, 1.0, 1732265768.9644074), (122.71, 1.0, 1732265769.3610733), (122.71, 1.0, 1732265805.0301569), (122.62, 1.0, 1732265827.6360407), (122.75, 1.0, 1732265943.3184645), (122.66, 1.0, 1732265991.3614163), (122.67, 1.0, 1732265996.4082358), (122.66, 1.0, 1732265996.4135077), (122.6, 3.0, 1732265996.4314103), (122.66, 1.0, 1732265997.5577695), (122.66, 1.0, 1732265997.55779), (122.65, 1.0, 1732265998.1754792), (122.66, 1.0, 1732265999.083375), (122.72, 1.0, 1732265999.0834591), (122.67, 1.0, 1732265999.1877983), (122.56, 1.0, 1732265999.4996765), (122.58, 1.0, 1732266006.7485716), (122.57, 1.0, 1732266009.251118), (122.53, 1.0, 1732266024.2405498), (122.57, 1.0, 1732266032.131626), (122.6, 6.0, 1732266208.0808618), (122.74, 2.0, 1732266331.3067567), (122.79, 1.0, 1732266405.7427144), (122.75, 1.0, 1732266426.695282), (122.71, 1.0, 1732266436.72379), (122.76, 1.0, 1732266466.3363936), (122.71, 2.0, 1732266475.993237), (122.68, 1.0, 1732266608.7935214), (122.68, 1.0, 1732266608.7936935), (122.68, 1.0, 1732266631.5296588), (122.67, 1.0, 1732266631.53586), (122.67, 1.0, 1732266640.2620394), (122.67, 1.0, 1732266642.8612118), (122.66, 1.0, 1732266643.43201), (122.68, 1.0, 1732266643.756405), (122.67, 1.0, 1732266652.3658688), (122.67, 1.0, 1732266652.38038), (122.53, 1.0, 1732266652.7398906), (122.51, 1.0, 1732266685.2507675), (122.39, 1.0, 1732266746.6866713), (122.35, 1.0, 1732266753.539492), (122.35, 1.0, 1732266756.647809), (122.39, 1.0, 1732266767.580395), (122.2, 1.0, 1732266767.8436587), (122.2, 1.0, 1732266775.5037646), (122.25, 1.0, 1732266779.6066203), (122.25, 1.0, 1732266779.6195576), (122.12, 1.0, 1732266779.6335576), (122.12, 1.0, 1732266822.739446), (122.19, 1.0, 1732266862.3182323), (122.15, 2.0, 1732266862.3183205), (122.15, 1.0, 1732266862.3183537), (122.11, 1.0, 1732266871.8702312), (121.99, 1.0, 1732266904.447415), (121.97, 1.0, 1732266904.6264627), (121.92, 1.0, 1732266908.0901406), (121.91, 1.0, 1732266908.0909784), (121.87, 1.0, 1732266908.1946428), (121.8, 1.0, 1732266911.1238668), (121.81, 1.0, 1732266918.2820747), (121.82, 1.0, 1732266918.2820747), (121.87, 2.0, 1732266943.0862117), (121.99, 1.0, 1732266943.207482), (122.0, 2.0, 1732266960.151926), (122.01, 1.0, 1732267138.5988164), (122.01, 1.0, 1732267138.599013), (122.05, 1.0, 1732267196.7511108), (122.25, 1.0, 1732267345.99666), (122.25, 1.0, 1732267346.0655882), (122.25, 1.0, 1732267346.1162837), (122.4, 2.0, 1732267398.2694712), (122.42, 2.0, 1732267398.2831805), (122.4, 1.0, 1732267417.3887277), (122.4, 2.0, 1732267425.9227102), (122.32, 1.0, 1732267480.5219283), (122.43, 1.0, 1732267502.4911947), (122.43, 1.0, 1732267502.5035796), (122.44, 1.0, 1732267502.588633), (122.43, 1.0, 1732267502.6014605), (122.43, 1.0, 1732267503.2776167), (122.54, 1.0, 1732267516.581103), (122.44, 2.0, 1732267607.5247722), (122.41, 1.0, 1732267607.554918), (122.43, 1.0, 1732267607.6575613), (122.43, 1.0, 1732267770.5433736), (122.36, 1.0, 1732267788.1932614), (122.36, 1.0, 1732267788.2082753), (122.36, 1.0, 1732267800.049539), (122.3, 1.0, 1732267800.664499), (122.27, 1.0, 1732267800.672102), (122.04, 1.0, 1732267861.6650956), (122.03, 1.0, 1732267861.7678556), (122.01, 5.0, 1732267989.4632044), (122.01, 1.0, 1732267989.4637008), (121.96, 3.0, 1732267989.477786), (121.96, 1.0, 1732267990.2702203), (121.96, 1.0, 1732267990.6953225), (122.02, 1.0, 1732267990.7974403), (122.03, 1.0, 1732267990.7974403), (121.87, 1.0, 1732267990.9760325), (122.05, 1.0, 1732267991.2900395), (121.87, 1.0, 1732267991.5750487), (121.95, 1.0, 1732268008.8966656), (121.95, 1.0, 1732268008.8967824), (121.95, 1.0, 1732268040.061282), (121.91, 1.0, 1732268052.3054688), (121.91, 1.0, 1732268052.305546), (121.91, 1.0, 1732268052.3055775), (121.91, 1.0, 1732268052.3086948), (121.85, 1.0, 1732268052.4284346), (121.95, 1.0, 1732268069.1265364), (121.95, 2.0, 1732268069.2515204), (121.95, 1.0, 1732268069.2680547), (121.95, 1.0, 1732268069.2680812), (121.95, 2.0, 1732268069.2802472), (121.95, 1.0, 1732268069.2933333), (121.95, 1.0, 1732268069.2933524), (121.9, 1.0, 1732268096.0968866), (121.95, 1.0, 1732268120.6756835), (121.95, 1.0, 1732268120.675712), (121.95, 1.0, 1732268120.689967), (121.95, 1.0, 1732268120.690019), (121.95, 1.0, 1732268120.7048287), (121.95, 1.0, 1732268120.7048633), (121.95, 1.0, 1732268120.717921), (121.95, 1.0, 1732268120.7179341), (121.95, 1.0, 1732268120.733884), (121.95, 1.0, 1732268120.733899), (121.95, 1.0, 1732268120.7497704), (121.95, 1.0, 1732268120.7497823), (121.95, 1.0, 1732268120.7614715), (122.1, 1.0, 1732268136.772164), (122.16, 1.0, 1732268181.5648863), (122.09, 2.0, 1732268275.2940667), (122.08, 1.0, 1732268275.66526), (122.03, 1.0, 1732268300.983496), (121.93, 1.0, 1732268331.49032), (121.84, 1.0, 1732268335.8977933), (121.81, 1.0, 1732268340.091102), (121.75, 1.0, 1732268340.0988867), (121.79, 1.0, 1732268340.1033132), (121.82, 1.0, 1732268355.728403), (121.83, 1.0, 1732268413.3485515), (121.84, 1.0, 1732268413.3485515), (121.94, 1.0, 1732268433.5431275), (121.94, 1.0, 1732268433.6785738), (121.94, 1.0, 1732268433.7918708), (121.94, 1.0, 1732268435.7422616), (121.94, 1.0, 1732268435.7461038), (121.83, 1.0, 1732268445.2475116), (121.82, 1.0, 1732268445.260635), (122.0, 1.0, 1732268560.6808403), (122.0, 1.0, 1732268560.6992705), (121.88, 1.0, 1732268569.0943806), (122.1, 1.0, 1732268795.4148967), (122.1, 1.0, 1732268795.4149792), (122.31, 2.0, 1732268805.50256), (122.5, 3.0, 1732268958.6342812), (122.5, 1.0, 1732268958.6343253), (122.5, 1.0, 1732268958.6344378), (122.5, 1.0, 1732268973.9148345), (122.5, 1.0, 1732268974.0964222), (122.49, 1.0, 1732269017.9504836), (122.48, 1.0, 1732269017.968994), (122.49, 1.0, 1732269018.047937), (122.48, 1.0, 1732269064.1951528), (122.6, 2.0, 1732269094.2574852), (122.6, 2.0, 1732269099.6905096), (122.6, 1.0, 1732269099.9665303), (122.6, 1.0, 1732269099.9667006), (122.6, 1.0, 1732269102.678671), (122.59, 1.0, 1732269114.1666667), (122.59, 2.0, 1732269117.3999453), (122.59, 2.0, 1732269120.2789204), (122.58, 1.0, 1732269123.2887485), (122.59, 1.0, 1732269126.0020056), (122.59, 1.0, 1732269127.3916862), (122.59, 1.0, 1732269129.3866315), (122.43, 1.0, 1732269140.5131905), (122.58, 2.0, 1732269140.65669), (122.42, 1.0, 1732269180.9369678), (122.42, 1.0, 1732269181.3918695), (122.58, 1.0, 1732269306.1630762), (122.5, 1.0, 1732269319.0531788), (122.52, 1.0, 1732269329.2052405), (122.47, 1.0, 1732269336.8359833), (122.57, 2.0, 1732269411.8705325), (122.57, 2.0, 1732269415.199773), (122.57, 1.0, 1732269418.575559), (122.57, 1.0, 1732269423.106088), (122.57, 1.0, 1732269424.0240767), (122.59, 1.0, 1732269427.7386973), (122.6, 1.0, 1732269433.027733), (122.47, 1.0, 1732269452.076572), (122.44, 2.0, 1732269521.8679752), (122.4, 1.0, 1732269557.8494775), (122.36, 1.0, 1732269634.0477257), (122.54, 1.0, 1732269680.1557412), (122.54, 1.0, 1732269680.155782), (122.75, 1.0, 1732269897.4968877), (122.85, 1.0, 1732269898.8422577), (122.86, 1.0, 1732269900.0766916), (122.88, 1.0, 1732269900.0766916), (122.9, 1.0, 1732269900.0766916), (122.89, 1.0, 1732269921.4590633), (122.89, 1.0, 1732269922.6754808), (122.89, 1.0, 1732269922.6755395), (122.9, 1.0, 1732269946.118588), (122.9, 1.0, 1732269946.1305065), (122.9, 1.0, 1732269947.0365736), (122.9, 1.0, 1732269947.3568552), (122.95, 1.0, 1732269953.2040863), (122.96, 1.0, 1732269956.3046837), (122.92, 1.0, 1732269956.615842), (123.0, 1.0, 1732269959.3473144), (123.0, 2.0, 1732269959.3473556), (123.0, 1.0, 1732269959.6733532), (123.0, 1.0, 1732269959.673394), (123.0, 1.0, 1732269960.1866844), (123.0, 1.0, 1732269963.699479), (122.98, 1.0, 1732269963.699658), (123.04, 1.0, 1732269963.701034), (123.05, 1.0, 1732269963.7010846), (123.0, 1.0, 1732269963.711943), (122.97, 1.0, 1732269963.7210722), (123.0, 1.0, 1732269963.741994), (123.14, 2.0, 1732270130.0436049), (123.15, 1.0, 1732270170.2663257), (123.06, 1.0, 1732270176.237602), (123.05, 1.0, 1732270176.2376733), (123.06, 1.0, 1732270176.2377548), (123.06, 1.0, 1732270176.237825), (123.06, 7.0, 1732270176.23785), (123.04, 1.0, 1732270176.2528508), (123.24, 1.0, 1732270214.572648), (123.25, 1.0, 1732270214.5745747), (123.33, 1.0, 1732270239.372998), (123.33, 1.0, 1732270239.594652), (123.35, 1.0, 1732270240.7354612), (123.42, 1.0, 1732270255.8090549), (123.42, 1.0, 1732270255.8091843), (123.45, 1.0, 1732270255.8240163), (123.52, 1.0, 1732270282.3659718), (123.52, 1.0, 1732270282.4021776), (123.52, 1.0, 1732270284.8629775), (123.52, 1.0, 1732270285.0340018), (123.52, 1.0, 1732270285.1451664), (123.43, 1.0, 1732270285.1589067), (123.52, 1.0, 1732270285.339064), (123.46, 1.0, 1732270287.9289231), (123.36, 1.0, 1732270301.7897046), (123.5, 1.0, 1732270302.692544), (123.48, 1.0, 1732270315.3588731), (123.37, 2.0, 1732270415.4632835), (123.5, 3.0, 1732270674.2094245), (123.47, 1.0, 1732270683.9015417), (123.37, 2.0, 1732270683.915434), (123.45, 1.0, 1732270783.3137057), (123.4, 1.0, 1732270794.673878), (123.4, 1.0, 1732270794.674709), (123.4, 1.0, 1732270794.6843076), (123.4, 1.0, 1732270794.6897113), (123.4, 1.0, 1732270794.6925085), (123.4, 1.0, 1732270794.8054404), (123.35, 1.0, 1732270795.070464), (123.35, 1.0, 1732270795.0705), (123.4, 1.0, 1732270796.2084072), (123.3, 1.0, 1732270812.1832087), (123.3, 1.0, 1732270815.3510234), (123.3, 1.0, 1732270842.7143545), (123.3, 1.0, 1732270843.0289588), (123.3, 1.0, 1732270853.067532), (123.3, 1.0, 1732270853.0676532), (123.3, 1.0, 1732270853.0702455), (123.3, 1.0, 1732270856.3828704), (123.3, 2.0, 1732270867.3132331), (123.3, 2.0, 1732270870.3169885), (123.22, 1.0, 1732270870.3313096), (123.3, 1.0, 1732270909.6608), (123.3, 2.0, 1732270909.6608455), (123.3, 1.0, 1732270911.4239943), (123.43, 1.0, 1732271135.1173098), (123.45, 1.0, 1732271168.2975116), (123.42, 1.0, 1732271177.5688038), (123.45, 1.0, 1732271187.4432893), (123.5, 1.0, 1732271194.823684), (123.52, 1.0, 1732271203.5524015), (123.52, 1.0, 1732271203.6057825), (123.52, 1.0, 1732271210.6515703), (123.59, 1.0, 1732271211.8520143), (123.59, 1.0, 1732271214.5548193), (123.63, 1.0, 1732271215.5402765), (123.63, 1.0, 1732271215.5403233), (123.57, 2.0, 1732271218.0128), (123.74, 1.0, 1732271240.0692422), (123.85, 2.0, 1732271267.8049543), (123.85, 1.0, 1732271268.0808837), (123.79, 1.0, 1732271289.265894), (123.8, 1.0, 1732271299.6213264), (123.76, 1.0, 1732271304.1033251), (123.76, 1.0, 1732271305.678508), (123.7, 2.0, 1732271325.1681073), (123.7, 1.0, 1732271332.1919756), (123.68, 1.0, 1732271332.4888592), (123.68, 1.0, 1732271332.790538), (123.59, 1.0, 1732271340.4238763), (123.5, 2.0, 1732271461.214353), (123.55, 1.0, 1732271499.4750843), (123.5, 1.0, 1732271557.5817845), (123.5, 1.0, 1732271576.1203623), (123.5, 1.0, 1732271630.8428154), (123.53, 1.0, 1732271630.8518755), (123.5, 1.0, 1732271637.056702), (123.47, 1.0, 1732271637.7025237), (123.46, 2.0, 1732271637.9450405), (123.46, 1.0, 1732271637.945184), (123.46, 2.0, 1732271637.945238), (123.44, 1.0, 1732271648.8551507), (123.5, 1.0, 1732271668.1129255), (123.5, 1.0, 1732271668.202162), (123.38, 1.0, 1732271779.30763), (123.38, 1.0, 1732271795.8043752), (123.55, 1.0, 1732271803.2586741), (123.38, 1.0, 1732271900.174648), (123.31, 2.0, 1732271903.4856083), (123.31, 1.0, 1732271903.566344), (123.31, 1.0, 1732271903.7791872), (123.31, 1.0, 1732271903.7913346), (123.31, 1.0, 1732271906.7346787), (123.41, 1.0, 1732271916.1431942), (123.31, 1.0, 1732271926.144252), (123.39, 1.0, 1732271928.3567235), (123.4, 2.0, 1732271931.5073843), (123.3, 1.0, 1732271941.2493258), (123.3, 1.0, 1732271941.2502215), (123.3, 1.0, 1732271941.2520106), (123.3, 1.0, 1732271941.2591944), (123.3, 1.0, 1732271941.2602277), (123.28, 1.0, 1732271941.3215995), (123.23, 1.0, 1732271945.925605), (123.19, 1.0, 1732271956.092239), (123.32, 1.0, 1732271962.0802188), (123.32, 1.0, 1732271963.06434), (123.32, 1.0, 1732271963.5554147), (123.21, 1.0, 1732271985.7342749), (123.21, 2.0, 1732271985.8869512), (123.21, 2.0, 1732271985.9334593), (123.16, 1.0, 1732272024.2209284), (123.15, 1.0, 1732272033.6933427), (123.15, 1.0, 1732272037.9410655), (123.15, 1.0, 1732272053.3778653), (123.15, 1.0, 1732272053.5656853), (123.16, 1.0, 1732272085.4642184), (123.16, 1.0, 1732272115.684833), (123.15, 1.0, 1732272115.6849055), (123.16, 1.0, 1732272115.711584), (123.16, 1.0, 1732272144.3071597), (123.16, 1.0, 1732272149.5372791), (123.16, 1.0, 1732272149.6416998), (123.25, 1.0, 1732272155.1163437), (123.18, 1.0, 1732272173.38638), (123.14, 1.0, 1732272177.5240812), (123.21, 1.0, 1732272196.2498589), (123.18, 1.0, 1732272316.9270039), (123.17, 1.0, 1732272332.3506255), (123.14, 1.0, 1732272343.8561776), (123.14, 1.0, 1732272344.1447928), (123.18, 1.0, 1732272413.991403), (123.2, 1.0, 1732272414.3171384), (123.1, 1.0, 1732272443.593262), (123.12, 1.0, 1732272446.6972861), (123.12, 1.0, 1732272466.114965), (123.12, 1.0, 1732272470.9231308), (123.13, 2.0, 1732272473.9833577), (123.12, 1.0, 1732272473.9833577), (123.12, 1.0, 1732272474.302816), (123.12, 1.0, 1732272474.3028655), (123.12, 1.0, 1732272474.3029096), (123.12, 1.0, 1732272491.5514205), (123.29, 1.0, 1732272552.24367), (123.19, 1.0, 1732272592.028869), (123.19, 1.0, 1732272592.2274876), (123.16, 1.0, 1732272675.22237), (123.16, 1.0, 1732272675.224914), (123.05, 1.0, 1732272675.233276), (123.0, 1.0, 1732272675.2437985), (123.05, 1.0, 1732272675.4125352), (123.16, 1.0, 1732272676.241802), (123.01, 1.0, 1732272690.2751048), (123.01, 1.0, 1732272690.2751553), (123.11, 1.0, 1732272762.7323437), (123.11, 1.0, 1732272764.7261624), (123.11, 1.0, 1732272778.8212757), (123.03, 3.0, 1732272785.8455813), (123.01, 2.0, 1732272785.8455813), (123.0, 1.0, 1732272785.8455813), (123.01, 1.0, 1732272785.8455813), (123.02, 1.0, 1732272785.8455813), (123.0, 1.0, 1732272785.8457978), (123.0, 1.0, 1732272785.845839), (123.01, 2.0, 1732272797.8633435), (123.0, 1.0, 1732272797.8633435), (123.2, 2.0, 1732272804.0936897), (123.08, 2.0, 1732272952.8134716), (123.08, 1.0, 1732272952.8135157), (123.08, 1.0, 1732272952.8135667), (123.08, 1.0, 1732272952.8242135), (123.08, 1.0, 1732272952.834273), (123.08, 1.0, 1732272952.834886), (123.08, 1.0, 1732272952.8455198), (123.08, 1.0, 1732272952.8560843), (123.08, 1.0, 1732272952.866649), (123.08, 5.0, 1732272952.866711), (123.18, 1.0, 1732272955.2557971), (123.18, 1.0, 1732272955.5765452), (123.18, 1.0, 1732272955.8479583), (123.09, 1.0, 1732272968.3332443), (123.09, 1.0, 1732272968.3333871), (123.08, 1.0, 1732272968.3337066), (123.08, 1.0, 1732272968.562677), (123.08, 1.0, 1732272969.270114), (123.02, 2.0, 1732273103.4777348), (123.0, 1.0, 1732273106.606645), (123.0, 1.0, 1732273106.607628), (123.0, 1.0, 1732273106.6833255), (123.0, 1.0, 1732273106.6833797), (123.0, 1.0, 1732273106.6834395), (123.0, 1.0, 1732273114.9068892), (123.04, 1.0, 1732273141.8295786), (123.05, 1.0, 1732273151.29216), (123.01, 1.0, 1732273176.6510248), (123.0, 1.0, 1732273176.652017), (122.98, 1.0, 1732273176.6682768), (122.98, 1.0, 1732273176.6683211), (122.98, 1.0, 1732273176.6790853), (122.98, 1.0, 1732273176.742517), (122.98, 1.0, 1732273176.7563698), (122.98, 1.0, 1732273189.6801224), (122.98, 1.0, 1732273189.7065487), (123.0, 1.0, 1732273189.7065969), (123.05, 1.0, 1732273220.5766191), (123.05, 1.0, 1732273220.5766585), (123.05, 2.0, 1732273220.5767026), (123.05, 1.0, 1732273220.8235965), (123.05, 1.0, 1732273221.613911), (123.05, 1.0, 1732273221.613944), (123.05, 1.0, 1732273221.6160612), (123.05, 1.0, 1732273221.6161249), (123.09, 1.0, 1732273221.641235), (123.08, 1.0, 1732273221.771593), (123.09, 1.0, 1732273222.8749013), (123.09, 1.0, 1732273223.7001216), (123.08, 1.0, 1732273270.6859663), (123.08, 1.0, 1732273290.0817788), (123.16, 1.0, 1732273301.7480533), (123.18, 1.0, 1732273301.7480533), (123.0, 1.0, 1732273360.950975), (122.9, 1.0, 1732273365.656803), (122.98, 1.0, 1732273368.7111473), (122.94, 1.0, 1732273368.7112515), (122.98, 1.0, 1732273368.753062), (122.98, 1.0, 1732273368.765675), (122.98, 1.0, 1732273368.778288), (122.98, 1.0, 1732273368.8070133), (122.98, 1.0, 1732273368.8198323), (122.98, 1.0, 1732273368.8322978), (122.98, 1.0, 1732273368.8457267), (122.96, 1.0, 1732273377.0669537), (122.94, 1.0, 1732273377.0720303), (122.96, 1.0, 1732273377.1771815), (122.99, 1.0, 1732273377.6425445), (122.99, 2.0, 1732273383.8944125), (122.99, 1.0, 1732273385.2404792), (122.99, 1.0, 1732273385.8802936), (122.75, 1.0, 1732273459.8859725), (122.75, 1.0, 1732273459.9603293), (122.8, 1.0, 1732273464.6079159), (122.8, 1.0, 1732273464.60996), (122.91, 1.0, 1732273516.2625227), (122.88, 1.0, 1732273529.316857), (122.84, 2.0, 1732273560.03187), (122.84, 1.0, 1732273560.166806), (122.8, 2.0, 1732273580.0276415), (122.86, 1.0, 1732273622.9525957), (122.8, 1.0, 1732273652.4397583), (122.8, 1.0, 1732273652.4397917), (122.8, 1.0, 1732273652.5414548), (122.75, 1.0, 1732273654.0808315), (122.75, 1.0, 1732273654.0987475), (122.75, 1.0, 1732273654.122718), (122.75, 1.0, 1732273654.653597), (122.72, 1.0, 1732273683.0739143), (122.74, 1.0, 1732273712.198867), (122.74, 1.0, 1732273736.776247), (122.71, 2.0, 1732273740.9765563), (122.5, 1.0, 1732273740.9765563), (122.59, 1.0, 1732273740.9765563), (122.62, 1.0, 1732273740.9765563), (122.66, 1.0, 1732273740.9765563), (122.67, 1.0, 1732273740.9765563), (122.68, 1.0, 1732273740.9765563), (122.7, 1.0, 1732273740.9765563), (122.47, 2.0, 1732273740.976752), (122.47, 1.0, 1732273740.9768713), (122.47, 2.0, 1732273740.9769104), (122.7, 1.0, 1732273740.977496), (122.7, 1.0, 1732273740.9780161), (122.7, 1.0, 1732273740.9909236), (122.7, 1.0, 1732273741.0035017), (122.61, 2.0, 1732273766.922574), (122.58, 1.0, 1732273766.922574), (122.6, 1.0, 1732273766.922574), (122.55, 2.0, 1732273766.9226887), (122.55, 1.0, 1732273766.922737), (122.55, 7.0, 1732273766.9227695), (122.77, 1.0, 1732273799.9152038), (122.62, 1.0, 1732273814.7858253), (122.63, 1.0, 1732273814.7858253), (122.78, 2.0, 1732273868.690287), (122.78, 1.0, 1732273889.0037303), (122.78, 1.0, 1732273889.0037303), (122.78, 2.0, 1732273915.4633095), (122.77, 1.0, 1732273954.0861526), (122.78, 1.0, 1732273954.0861526), (122.73, 1.0, 1732273954.0863373), (122.78, 1.0, 1732273954.0881133), (122.78, 1.0, 1732273954.0885944), (122.78, 2.0, 1732273954.1021292), (122.78, 1.0, 1732273972.7643895), (122.78, 3.0, 1732273980.0786464), (122.79, 1.0, 1732273980.0786915), (122.79, 1.0, 1732273980.2718947), (122.78, 2.0, 1732273980.2861116), (123.01, 1.0, 1732274068.8919578), (122.88, 1.0, 1732274139.0550745), (122.92, 1.0, 1732274162.2381787), (122.92, 1.0, 1732274162.5253243), (122.89, 1.0, 1732274166.4911776), (122.89, 1.0, 1732274174.616449), (122.93, 1.0, 1732274214.644252), (123.08, 1.0, 1732274314.938315), (123.12, 1.0, 1732274314.9482694), (123.07, 1.0, 1732274315.1383564), (123.08, 1.0, 1732274315.7415512), (123.17, 3.0, 1732274353.6854255), (123.16, 1.0, 1732274373.563059), (123.27, 1.0, 1732274378.0889363), (123.24, 1.0, 1732274378.0892687), (123.23, 1.0, 1732274420.804496), (123.23, 2.0, 1732274421.008401), (123.23, 1.0, 1732274421.0234983), (123.23, 1.0, 1732274421.0235946), (123.23, 1.0, 1732274421.1798992), (123.1, 1.0, 1732274425.0644476), (123.22, 1.0, 1732274441.859193), (123.22, 1.0, 1732274441.859193), (123.23, 1.0, 1732274441.859193), (123.11, 1.0, 1732274441.873026), (123.11, 1.0, 1732274441.873077), (123.07, 1.0, 1732274449.9713092), (123.1, 1.0, 1732274450.5378048), (123.07, 1.0, 1732274452.7458549), (123.18, 1.0, 1732274454.5958016), (123.18, 1.0, 1732274454.5969543), (123.07, 1.0, 1732274456.4061823), (123.09, 1.0, 1732274468.588148), (123.0, 2.0, 1732274497.6546867), (123.0, 2.0, 1732274497.6680071), (123.0, 1.0, 1732274501.8582428), (123.0, 1.0, 1732274501.8582428), (122.84, 1.0, 1732274569.615376), (122.9, 1.0, 1732274570.8190591), (122.9, 1.0, 1732274570.8190591), (122.86, 1.0, 1732274570.8341615), (122.9, 1.0, 1732274570.892524), (122.9, 1.0, 1732274570.8931174), (122.9, 1.0, 1732274596.304855), (122.9, 1.0, 1732274612.2881172), (122.9, 2.0, 1732274612.3663905), (122.9, 1.0, 1732274612.525874), (122.9, 1.0, 1732274612.7753165), (122.9, 1.0, 1732274613.1411364), (122.9, 1.0, 1732274613.7036462), (122.9, 1.0, 1732274617.4222794), (122.9, 1.0, 1732274617.4224253), (122.9, 1.0, 1732274617.500559), (122.9, 1.0, 1732274617.5005803), (122.9, 1.0, 1732274617.5132315), (122.9, 1.0, 1732274617.513248), (122.8, 1.0, 1732274658.5446112), (122.8, 1.0, 1732274658.5446737), (122.91, 1.0, 1732274682.6118317), (122.83, 1.0, 1732274773.1360288), (122.83, 1.0, 1732274773.1361032), (122.8, 1.0, 1732274841.2862031), (122.8, 1.0, 1732274843.618575), (122.81, 2.0, 1732274846.8473513), (122.75, 1.0, 1732274912.7355952), (122.75, 1.0, 1732274912.7356584), (122.75, 1.0, 1732274912.7726183), (122.75, 1.0, 1732274914.4883955), (122.75, 1.0, 1732274917.0402253), (122.75, 1.0, 1732274917.0472083), (122.75, 1.0, 1732274917.0588715), (122.75, 1.0, 1732274919.4085033), (122.72, 1.0, 1732274928.6792235), (122.75, 1.0, 1732274940.2097323), (122.75, 2.0, 1732274951.652508), (122.75, 2.0, 1732274951.8286452), (122.75, 2.0, 1732274954.1083832), (122.75, 2.0, 1732274989.4670408), (122.75, 1.0, 1732274991.2292087), (122.74, 1.0, 1732275001.0477436), (122.75, 1.0, 1732275001.140334), (122.75, 2.0, 1732275001.1527655), (122.65, 1.0, 1732275063.5703752), (122.65, 1.0, 1732275123.7072089), (122.65, 1.0, 1732275123.7073145), (122.65, 1.0, 1732275123.7073648), (122.65, 1.0, 1732275123.708244), (122.65, 1.0, 1732275123.710094), (122.65, 1.0, 1732275123.7116768), (122.65, 1.0, 1732275123.9181824), (122.6, 1.0, 1732275139.186499), (122.6, 1.0, 1732275139.2972963), (122.6, 1.0, 1732275139.3092446), (122.6, 1.0, 1732275139.3209476), (122.6, 1.0, 1732275139.333347), (122.6, 1.0, 1732275139.344879), (122.6, 1.0, 1732275139.3564205), (122.45, 1.0, 1732275151.2813983), (122.33, 1.0, 1732275192.0785837), (122.33, 1.0, 1732275192.2820094), (122.33, 1.0, 1732275192.282065), (122.33, 1.0, 1732275192.530289), (122.33, 1.0, 1732275192.5303679), (122.33, 1.0, 1732275192.631735), (122.33, 1.0, 1732275192.6317644), (122.33, 1.0, 1732275192.9767623), (122.33, 1.0, 1732275198.5143108), (122.23, 1.0, 1732275200.5940208), (122.24, 1.0, 1732275200.5940208), (122.32, 1.0, 1732275201.571129), (122.33, 1.0, 1732275201.5725148), (122.33, 1.0, 1732275201.5729823), (122.33, 1.0, 1732275221.4996169), (122.33, 1.0, 1732275221.4996927), (122.31, 1.0, 1732275250.6353188), (122.31, 1.0, 1732275250.6353836), (122.2, 1.0, 1732275309.0994463), (122.2, 1.0, 1732275309.0995302), (122.2, 1.0, 1732275309.0996027), (122.2, 1.0, 1732275309.0996552), (122.2, 1.0, 1732275309.099687), (122.33, 1.0, 1732275320.3129284), (122.33, 1.0, 1732275320.3129675), (122.2, 1.0, 1732275377.3363743), (122.2, 1.0, 1732275377.3364067), (122.2, 3.0, 1732275377.3364308), (122.35, 1.0, 1732275449.7537956), (122.18, 1.0, 1732275455.1687212), (122.13, 1.0, 1732275470.2775974), (122.11, 1.0, 1732275494.064356), (122.11, 1.0, 1732275494.064446), (122.11, 1.0, 1732275509.4364407), (122.11, 1.0, 1732275509.4493384), (122.11, 1.0, 1732275509.461597), (122.11, 1.0, 1732275509.461612), (122.11, 2.0, 1732275541.2979655), (122.1, 2.0, 1732275603.1826687), (122.1, 1.0, 1732275627.4809685), (122.11, 1.0, 1732275627.4809685), (122.11, 1.0, 1732275627.4831088), (122.12, 1.0, 1732275627.505025), (122.12, 1.0, 1732275627.5050745), (122.11, 1.0, 1732275627.5213244), (122.11, 1.0, 1732275627.5214455), (122.11, 1.0, 1732275627.667035), (122.11, 1.0, 1732275627.668122), (122.07, 1.0, 1732275627.9029706), (122.0, 1.0, 1732275756.612579), (121.92, 1.0, 1732275777.8372977), (121.92, 1.0, 1732275778.18812), (121.91, 1.0, 1732275778.3972216), (121.92, 1.0, 1732275778.5283713), (121.91, 1.0, 1732275778.662603), (121.85, 1.0, 1732275837.1451285), (121.85, 1.0, 1732275837.3317478), (121.85, 1.0, 1732275837.352523), (121.85, 1.0, 1732275837.3640523), (121.85, 1.0, 1732275837.3754752), (121.8, 1.0, 1732275853.8413029), (121.8, 1.0, 1732275853.8554451), (121.8, 1.0, 1732275854.9837446), (121.77, 1.0, 1732275854.983787), (121.77, 1.0, 1732275854.9894395), (121.77, 1.0, 1732275855.1678395), (121.76, 1.0, 1732275856.1739123), (121.76, 1.0, 1732275857.7700665), (121.84, 1.0, 1732275877.6761985), (121.76, 1.0, 1732275879.415124), (121.76, 1.0, 1732275880.225082), (121.76, 1.0, 1732275886.4750469), (121.72, 1.0, 1732275924.450596), (121.73, 1.0, 1732275924.450596), (121.71, 1.0, 1732275927.6704812), (121.74, 1.0, 1732276027.7779996), (121.8, 1.0, 1732276136.7540183), (121.84, 1.0, 1732276149.0883048), (121.88, 1.0, 1732276149.0885594), (121.84, 1.0, 1732276149.408445), (122.0, 1.0, 1732276216.8983767), (122.01, 1.0, 1732276216.8983767), (121.91, 1.0, 1732276227.1234753), (121.91, 2.0, 1732276231.194807), (121.91, 1.0, 1732276231.95019), (122.0, 1.0, 1732276324.7888005), (121.87, 2.0, 1732276324.8025506), (122.03, 1.0, 1732276329.5594826), (122.03, 1.0, 1732276329.5596352), (122.01, 2.0, 1732276341.1957927), (122.0, 1.0, 1732276347.6584), (122.01, 1.0, 1732276347.6584), (122.15, 1.0, 1732276523.7570577), (122.19, 1.0, 1732276523.953544), (122.13, 1.0, 1732276677.1866481), (122.13, 1.0, 1732276830.6147068), (121.95, 1.0, 1732276862.6207173), (121.9, 1.0, 1732276962.350515), (121.9, 2.0, 1732276962.3506236), (121.99, 1.0, 1732277065.9382966), (122.09, 1.0, 1732277072.4028916), (122.1, 1.0, 1732277072.4034195), (122.11, 1.0, 1732277072.4051983), (122.14, 1.0, 1732277072.5747976), (122.11, 1.0, 1732277084.7434428), (122.23, 1.0, 1732277137.4733396), (122.0, 1.0, 1732277212.97869), (122.09, 2.0, 1732277290.9043424), (122.0, 1.0, 1732277316.1224463), (122.0, 1.0, 1732277316.2741268), (121.98, 1.0, 1732277444.332037), (121.86, 1.0, 1732277511.0230875), (121.85, 1.0, 1732277517.9016771), (121.82, 1.0, 1732277517.9018211), (121.86, 1.0, 1732277542.4422705), (122.11, 1.0, 1732277597.7618368), (122.12, 1.0, 1732277597.7618368), (122.1, 1.0, 1732277680.233961), (122.25, 1.0, 1732277751.1900585), (122.29, 1.0, 1732277817.433746), (122.29, 2.0, 1732277817.4337864), (122.47, 1.0, 1732277848.1543314), (122.48, 1.0, 1732277857.4178784), (122.47, 1.0, 1732277884.5480204), (122.58, 2.0, 1732277904.6247556), (122.34, 1.0, 1732277942.778012), (122.32, 1.0, 1732278056.4771008), (122.51, 1.0, 1732278058.04778), (122.53, 1.0, 1732278211.474893), (122.54, 1.0, 1732278211.474893), (122.3, 1.0, 1732278313.489322), (122.52, 1.0, 1732278364.9080431), (122.31, 2.0, 1732278411.7708032), (122.31, 1.0, 1732278411.8908055), (122.31, 1.0, 1732278411.8908849), (122.26, 1.0, 1732278441.9970672), (122.27, 1.0, 1732278441.9970672), (122.54, 1.0, 1732278518.3365676), (122.54, 1.0, 1732278518.3365676), (122.44, 1.0, 1732278519.7664845), (122.47, 1.0, 1732278519.7664845), (122.5, 1.0, 1732278534.6375356), (122.59, 1.0, 1732278534.6519673), (122.59, 1.0, 1732278534.6525424), (122.48, 1.0, 1732278555.120721), (122.47, 2.0, 1732278559.937275), (122.56, 1.0, 1732278595.031719), (122.51, 1.0, 1732278616.378087), (122.71, 1.0, 1732278729.545297), (122.74, 1.0, 1732278729.545297), (122.75, 1.0, 1732278729.545297), (122.74, 1.0, 1732278729.6244972), (122.75, 1.0, 1732278729.6245377), (122.6, 1.0, 1732278730.288011), (122.65, 1.0, 1732278863.216818), (122.65, 1.0, 1732278870.6256895), (122.65, 1.0, 1732278885.6288323), (122.56, 1.0, 1732278935.9320548), (122.51, 1.0, 1732279025.8714747), (122.44, 1.0, 1732279175.3591533), (122.39, 1.0, 1732279175.437329), (122.54, 1.0, 1732279301.9139109), (122.5, 1.0, 1732279320.0100148), (122.53, 1.0, 1732279348.1981163), (122.4, 1.0, 1732279470.7312179), (122.55, 4.0, 1732279564.6467462), (122.5, 1.0, 1732279590.19885), (122.49, 1.0, 1732279621.801296), (122.5, 1.0, 1732279642.4021094), (122.52, 1.0, 1732279688.4293337), (122.51, 1.0, 1732279691.8186986), (122.51, 1.0, 1732279694.877508), (122.51, 1.0, 1732279694.9413831), (122.51, 1.0, 1732279694.972836), (122.51, 1.0, 1732279699.0602436), (122.47, 2.0, 1732279734.5794992), (122.52, 1.0, 1732279734.7683113), (122.45, 1.0, 1732279736.010506), (122.4, 1.0, 1732279762.357443), (122.4, 1.0, 1732279762.358912), (122.4, 1.0, 1732279763.0154328), (122.4, 1.0, 1732279763.015475), (122.4, 1.0, 1732279765.9206257), (122.4, 1.0, 1732279765.9207447), (122.48, 2.0, 1732279806.780655), (122.48, 1.0, 1732279806.7808897), (122.38, 1.0, 1732279835.2272892), (122.39, 1.0, 1732279835.2272892), (122.41, 1.0, 1732279915.6237435), (122.4, 1.0, 1732279915.640175), (122.35, 1.0, 1732279916.6093347), (122.3, 1.0, 1732279921.8856256), (122.22, 1.0, 1732279928.034544), (122.04, 1.0, 1732280041.548421), (122.04, 2.0, 1732280050.994566), (122.0, 1.0, 1732280050.994566), (122.03, 1.0, 1732280050.994566), (122.0, 5.0, 1732280061.7973633), (122.0, 5.0, 1732280064.8823514), (122.0, 1.0, 1732280066.5277565), (121.64, 1.0, 1732280269.488588), (121.6, 4.0, 1732280269.4886975), (121.6, 5.0, 1732280274.7196755), (121.6, 1.0, 1732280308.4350712), (121.6, 1.0, 1732280315.5804524), (121.6, 1.0, 1732280315.5807936), (121.6, 2.0, 1732280315.5808928), (121.65, 1.0, 1732280315.5811925), (121.7, 2.0, 1732280355.299574), (121.63, 1.0, 1732280385.9265597), (121.51, 1.0, 1732280395.949458), (121.5, 1.0, 1732280395.9525497), (121.59, 1.0, 1732280396.029178), (121.37, 1.0, 1732280409.6554143), (121.38, 1.0, 1732280409.6554143), (121.42, 1.0, 1732280419.3406236), (121.38, 1.0, 1732280419.868493), (121.36, 1.0, 1732280425.8515356), (121.52, 1.0, 1732280449.477347), (121.27, 2.0, 1732280462.6546035), (121.39, 2.0, 1732280548.8797739), (121.35, 3.0, 1732280580.4879887), (121.33, 1.0, 1732280626.844008), (121.34, 1.0, 1732280626.844008), (121.34, 2.0, 1732280631.598692), (121.29, 1.0, 1732280733.8853724), (121.45, 1.0, 1732280737.7026036), (121.45, 1.0, 1732280737.7036676), (121.45, 1.0, 1732280737.7047164), (121.45, 1.0, 1732280737.7057648), (121.45, 1.0, 1732280737.7128437), (121.45, 1.0, 1732280737.7133243), (121.45, 1.0, 1732280737.72333), (121.45, 1.0, 1732280737.7347736), (121.45, 1.0, 1732280737.7401133), (121.45, 1.0, 1732280737.7484963), (121.34, 1.0, 1732280742.8800788), (121.4, 1.0, 1732280750.2763476), (121.41, 1.0, 1732280755.9446464), (121.4, 1.0, 1732280758.8013701), (121.4, 1.0, 1732280758.8022048), (121.4, 1.0, 1732280760.1502805), (121.4, 1.0, 1732280761.9776616), (121.4, 1.0, 1732280763.2865005), (121.4, 1.0, 1732280766.8268735), (121.4, 1.0, 1732280769.6753333), (121.4, 1.0, 1732280769.9551785), (121.41, 1.0, 1732280777.183544), (121.54, 1.0, 1732280784.9754462), (121.54, 1.0, 1732280784.9766195), (121.54, 1.0, 1732280784.9770768), (121.43, 1.0, 1732280787.3794808), (121.44, 1.0, 1732280788.6360087), (121.43, 1.0, 1732280788.6507504), (121.43, 3.0, 1732280789.9879863), (121.49, 1.0, 1732280802.1062703), (121.4, 2.0, 1732280811.3521786), (121.44, 1.0, 1732280898.3244803), (121.44, 1.0, 1732280898.325297), (121.44, 1.0, 1732280898.3351767), (121.44, 1.0, 1732280909.3681717), (121.44, 1.0, 1732280909.3830192), (121.45, 1.0, 1732280917.272934), (121.44, 1.0, 1732280937.3867183), (121.44, 1.0, 1732280937.4095457), (121.44, 1.0, 1732280937.491598), (121.44, 1.0, 1732280938.466736), (121.44, 1.0, 1732280938.9587576), (121.44, 1.0, 1732280940.0430012), (121.4, 2.0, 1732280943.055485), (121.4, 1.0, 1732280946.0879834), (121.4, 1.0, 1732280946.0880487), (121.46, 1.0, 1732280993.3032343), (121.44, 2.0, 1732281073.2571564), (121.45, 1.0, 1732281075.3308325), (121.45, 1.0, 1732281075.6757321), (121.45, 1.0, 1732281075.98802), (121.46, 1.0, 1732281099.8506227), (121.41, 1.0, 1732281099.9266467), (121.41, 1.0, 1732281099.9267724), (121.44, 1.0, 1732281102.9142506), (121.4, 1.0, 1732281120.696602), (121.4, 2.0, 1732281120.7794292), (121.4, 1.0, 1732281120.780462), (121.4, 1.0, 1732281135.91437), (121.4, 1.0, 1732281136.533956), (121.35, 2.0, 1732281138.9048424), (121.35, 1.0, 1732281139.0091648), (121.35, 1.0, 1732281139.2230139), (121.4, 1.0, 1732281139.3510892), (121.4, 1.0, 1732281148.4414334), (121.35, 1.0, 1732281176.625613), (121.35, 1.0, 1732281176.904946), (121.35, 1.0, 1732281211.3041728), (121.35, 1.0, 1732281213.1026185), (121.35, 1.0, 1732281213.1211452), (121.35, 1.0, 1732281213.1217604), (121.35, 1.0, 1732281213.1237187), (121.35, 1.0, 1732281213.1341496), (121.42, 1.0, 1732281287.2133882), (121.42, 1.0, 1732281287.213489), (121.48, 1.0, 1732281287.7250645), (121.49, 1.0, 1732281287.7250645), (121.36, 1.0, 1732281356.0098171), (121.43, 1.0, 1732281435.5268464), (121.44, 1.0, 1732281437.6381516), (121.44, 1.0, 1732281464.2002656), (121.44, 1.0, 1732281502.4805872), (121.43, 1.0, 1732281512.9130995), (121.41, 1.0, 1732281513.1241484), (121.41, 1.0, 1732281513.636814), (121.39, 1.0, 1732281554.1415389), (121.38, 1.0, 1732281583.81619), (121.35, 1.0, 1732281600.6225674), (121.46, 1.0, 1732281658.395602), (121.48, 1.0, 1732281658.6050804), (121.53, 1.0, 1732281671.7895517), (121.49, 1.0, 1732281672.00944), (121.49, 1.0, 1732281672.8807762), (121.48, 1.0, 1732281672.880828), (121.67, 1.0, 1732281731.063383), (121.61, 1.0, 1732281731.3539703), (121.61, 2.0, 1732281778.260918), (121.62, 1.0, 1732281779.5959015), (121.5, 1.0, 1732281867.7854624), (121.5, 1.0, 1732281867.7953868), (121.5, 1.0, 1732281867.7954996), (121.5, 1.0, 1732281867.7955475), (121.5, 1.0, 1732281867.7980647), (121.5, 1.0, 1732281867.8839903), (121.5, 1.0, 1732281867.887362), (121.5, 1.0, 1732281867.887921), (121.5, 1.0, 1732281867.9397073), (121.5, 1.0, 1732281868.2264235), (121.5, 1.0, 1732281869.190451), (121.34, 1.0, 1732281882.284295), (121.3, 1.0, 1732281882.4850328), (121.3, 1.0, 1732281882.4980972), (121.3, 1.0, 1732281885.7358143), (121.4, 1.0, 1732281892.0247376), (121.41, 1.0, 1732281892.0247989), (121.37, 2.0, 1732281893.3517036), (121.37, 1.0, 1732281893.3884716), (121.37, 1.0, 1732281893.5882668), (121.37, 1.0, 1732281893.6068563), (121.37, 1.0, 1732281893.6184344), (121.37, 1.0, 1732281893.630386), (121.37, 1.0, 1732281893.641893), (121.37, 1.0, 1732281893.6540976), (121.37, 1.0, 1732281893.665629), (121.46, 1.0, 1732281907.0102344), (121.46, 1.0, 1732281910.5474608), (121.43, 1.0, 1732281910.547504), (121.48, 1.0, 1732282026.6198692), (121.49, 1.0, 1732282026.6198692), (121.35, 1.0, 1732282083.8838685), (121.35, 1.0, 1732282083.8839147), (121.35, 1.0, 1732282083.9372172), (121.45, 1.0, 1732282128.7137463), (121.5, 1.0, 1732282174.4038725), (121.34, 1.0, 1732282203.5788677), (121.34, 1.0, 1732282203.6311226), (121.39, 1.0, 1732282205.766266), (121.39, 1.0, 1732282205.778241), (121.38, 1.0, 1732282206.6845725), (121.35, 1.0, 1732282264.224944), (121.35, 1.0, 1732282264.2250412), (121.35, 1.0, 1732282264.2251515), (121.34, 1.0, 1732282264.2252839), (121.33, 1.0, 1732282284.7287345), (121.29, 1.0, 1732282293.2222707), (121.26, 1.0, 1732282293.9206128), (121.26, 1.0, 1732282294.1337829), (121.26, 1.0, 1732282294.135718), (121.26, 1.0, 1732282294.1443243), (121.26, 1.0, 1732282294.6407056), (121.21, 1.0, 1732282297.8379796), (121.26, 1.0, 1732282297.8380792), (121.25, 1.0, 1732282315.5571415), (121.2, 1.0, 1732282337.258465), (121.1, 1.0, 1732282362.339775), (121.09, 1.0, 1732282362.6238256), (121.09, 1.0, 1732282372.8474884), (121.07, 1.0, 1732282372.943246), (121.11, 1.0, 1732282380.1293092), (121.06, 1.0, 1732282384.744239), (121.0, 1.0, 1732282387.4399843), (120.96, 2.0, 1732282414.5915852), (120.88, 1.0, 1732282428.8137915), (120.88, 1.0, 1732282428.980579), (120.88, 1.0, 1732282432.518504), (120.87, 1.0, 1732282446.5850081), (120.94, 1.0, 1732282585.892184), (120.98, 1.0, 1732282585.9707177), (121.2, 1.0, 1732282617.7457497), (121.2, 1.0, 1732282622.22891), (121.2, 1.0, 1732282624.728676), (121.2, 1.0, 1732282627.7290723), (121.2, 1.0, 1732282631.3394983), (121.2, 1.0, 1732282656.8742561), (121.2, 1.0, 1732282656.896506), (121.2, 1.0, 1732282656.9172573), (121.2, 1.0, 1732282656.9534764), (121.2, 1.0, 1732282657.1917028), (121.2, 1.0, 1732282657.217074), (121.2, 1.0, 1732282668.6158667), (121.2, 1.0, 1732282668.615963), (121.11, 2.0, 1732282715.6517675), (121.2, 1.0, 1732282762.5546415), (121.25, 1.0, 1732282762.5559487), (121.2, 1.0, 1732282762.577268), (121.2, 1.0, 1732282762.604951), (121.2, 1.0, 1732282762.8184533), (121.19, 1.0, 1732282762.8442712), (121.2, 1.0, 1732282762.8442948), (121.2, 1.0, 1732282763.7880836), (121.2, 1.0, 1732282763.964824), (121.2, 1.0, 1732282764.5816271), (121.2, 1.0, 1732282765.1611753), (121.2, 1.0, 1732282765.2125556), (121.2, 1.0, 1732282765.2384453), (121.2, 1.0, 1732282765.256811), (121.2, 1.0, 1732282765.6274548), (121.2, 1.0, 1732282771.6637764), (121.2, 1.0, 1732282791.4090981), (121.2, 1.0, 1732282791.4348445), (121.2, 1.0, 1732282791.4575448), (121.2, 1.0, 1732282791.487925), (121.2, 1.0, 1732282791.5153253), (121.2, 1.0, 1732282791.534604), (121.2, 1.0, 1732282791.5579576), (121.2, 1.0, 1732282793.581749), (121.21, 1.0, 1732282793.817247), (121.21, 1.0, 1732282793.8383408), (121.2, 1.0, 1732282793.881956), (121.2, 1.0, 1732282794.385394), (121.2, 1.0, 1732282794.6459162), (121.29, 1.0, 1732282795.0341892), (121.2, 1.0, 1732282795.0905137), (121.2, 1.0, 1732282795.324884), (121.2, 1.0, 1732282795.4678373), (121.2, 1.0, 1732282795.4869692), (121.2, 1.0, 1732282795.5074008), (121.2, 1.0, 1732282795.5632477), (121.2, 1.0, 1732282795.5824206), (121.2, 1.0, 1732282795.601127), (121.2, 1.0, 1732282795.6236482), (121.2, 1.0, 1732282795.645336), (121.2, 1.0, 1732282797.4724493), (121.2, 1.0, 1732282798.4608753), (121.2, 1.0, 1732282798.645058), (121.2, 1.0, 1732282798.778813), (121.2, 1.0, 1732282799.492616), (121.2, 1.0, 1732282799.7570815), (121.2, 1.0, 1732282800.0248623), (121.2, 1.0, 1732282803.0279877), (121.2, 1.0, 1732282803.1064875), (121.2, 1.0, 1732282803.151289), (121.2, 1.0, 1732282806.3357065), (121.2, 1.0, 1732282819.6428168), (121.2, 1.0, 1732282819.7458808), (121.2, 1.0, 1732282827.4540503), (121.2, 1.0, 1732282840.3996115), (121.25, 1.0, 1732282840.3996625), (121.2, 1.0, 1732282840.4689565), (121.2, 1.0, 1732282840.508371), (121.2, 1.0, 1732282841.0521815), (121.2, 1.0, 1732282841.217899), (121.2, 1.0, 1732282841.3591313), (121.2, 1.0, 1732282841.632498), (121.2, 1.0, 1732282843.4005175), (121.2, 1.0, 1732282847.6776328), (121.25, 1.0, 1732282847.6777432), (121.2, 4.0, 1732282847.7260182), (121.2, 1.0, 1732282847.9445355), (121.2, 1.0, 1732282849.7679012), (121.2, 1.0, 1732282849.9224231), (121.2, 1.0, 1732282849.944944), (121.2, 1.0, 1732282849.968758), (121.2, 1.0, 1732282849.9912372), (121.2, 1.0, 1732282850.7045777), (121.2, 1.0, 1732282853.7119515), (121.2, 1.0, 1732282856.7121632), (121.2, 1.0, 1732282859.3799205), (121.2, 1.0, 1732282859.4015048), (121.2, 1.0, 1732282859.4224007), (121.2, 1.0, 1732282859.4351068), (121.2, 1.0, 1732282859.4428077), (121.2, 1.0, 1732282859.5003064), (121.2, 1.0, 1732282860.045349), (121.2, 1.0, 1732282863.0568595), (121.2, 1.0, 1732282865.4854999), (121.2, 1.0, 1732282871.0485463), (121.2, 1.0, 1732282879.0998104), (121.2, 1.0, 1732282883.967163), (121.2, 1.0, 1732282885.9360285), (121.2, 1.0, 1732282886.9835517), (121.2, 1.0, 1732282889.542752), (121.24, 1.0, 1732282889.5428476), (121.32, 1.0, 1732282913.310238), (121.16, 1.0, 1732282987.3932376), (121.16, 1.0, 1732282987.3933613), (121.16, 1.0, 1732282989.0066385), (121.16, 1.0, 1732282989.006664), (121.13, 2.0, 1732283106.7666123), (121.2, 1.0, 1732283184.668094), (121.2, 1.0, 1732283207.3769057), (121.2, 1.0, 1732283207.555702), (121.2, 1.0, 1732283207.5688024), (121.2, 1.0, 1732283207.5834315), (121.25, 1.0, 1732283208.8691716), (121.25, 1.0, 1732283213.0921783), (121.21, 1.0, 1732283350.5566928), (121.21, 1.0, 1732283350.556774), (121.19, 1.0, 1732283350.8309777), (121.19, 1.0, 1732283350.8410635), (121.18, 1.0, 1732283350.8411267), (121.19, 1.0, 1732283350.8544188), (121.18, 1.0, 1732283351.469324), (121.22, 1.0, 1732283464.4047456), (121.34, 1.0, 1732283504.4335916), (121.35, 1.0, 1732283504.4335916), (121.21, 1.0, 1732283507.450156), (121.1, 1.0, 1732283522.134496), (121.09, 1.0, 1732283523.036849), (121.09, 1.0, 1732283523.3224175), (121.09, 1.0, 1732283523.574852), (121.09, 1.0, 1732283523.5758524), (121.25, 1.0, 1732283640.0204475), (121.27, 1.0, 1732283651.8064976), (121.36, 2.0, 1732283652.2133615), (121.16, 1.0, 1732283799.9934547), (121.21, 1.0, 1732283806.8316734), (121.2, 1.0, 1732283912.2574246), (121.2, 1.0, 1732283914.8991275), (121.21, 1.0, 1732283916.2148411), (121.3, 1.0, 1732283927.6076996), (121.37, 1.0, 1732283947.7774851), (121.38, 1.0, 1732283947.7774851), (121.39, 1.0, 1732283958.6345944), (121.32, 1.0, 1732283963.8468604), (121.31, 1.0, 1732283963.8481863), (121.21, 1.0, 1732283982.9048717), (121.47, 1.0, 1732284095.555836), (121.27, 1.0, 1732284105.3772004), (121.27, 1.0, 1732284105.4045422), (121.26, 1.0, 1732284105.4324512), (121.3, 1.0, 1732284147.4896264), (121.3, 4.0, 1732284147.4896894), (121.3, 1.0, 1732284164.4933615), (121.3, 1.0, 1732284164.493434), (121.3, 1.0, 1732284164.4934795), (121.3, 2.0, 1732284164.4935124), (121.42, 1.0, 1732284175.3830965), (121.43, 1.0, 1732284175.3854735), (121.43, 1.0, 1732284175.3968165), (121.54, 1.0, 1732284180.037212), (121.58, 1.0, 1732284187.969162), (121.56, 1.0, 1732284243.3368237), (121.58, 1.0, 1732284243.3368237), (121.28, 1.0, 1732284303.1551857), (121.29, 1.0, 1732284303.1551857), (121.25, 1.0, 1732284303.23367), (121.25, 1.0, 1732284388.596712), (121.25, 1.0, 1732284388.5981388), (121.25, 1.0, 1732284388.5982091), (121.25, 1.0, 1732284388.598261), (121.19, 1.0, 1732284402.2275944), (121.15, 1.0, 1732284454.9909546), (121.23, 1.0, 1732284502.839452), (121.24, 1.0, 1732284502.917614), (121.23, 1.0, 1732284503.5006468), (121.18, 1.0, 1732284503.6592636), (121.28, 1.0, 1732284504.4190812), (121.23, 1.0, 1732284522.7351248), (121.23, 1.0, 1732284522.7351816), (121.09, 1.0, 1732284552.7975197), (121.06, 1.0, 1732284580.2626193), (120.93, 1.0, 1732284580.3464057), (120.96, 1.0, 1732284581.004896), (121.0, 1.0, 1732284581.296801), (121.0, 1.0, 1732284581.2968693), (121.0, 2.0, 1732284582.070974), (120.98, 1.0, 1732284582.2495437), (120.99, 1.0, 1732284686.6795204), (121.0, 1.0, 1732284686.6795204), (121.0, 1.0, 1732284766.108236), (121.0, 1.0, 1732284766.189597), (121.0, 1.0, 1732284768.737682), (121.0, 1.0, 1732284768.819778), (121.05, 1.0, 1732284796.934386), (121.05, 4.0, 1732284804.422123), (120.96, 1.0, 1732284820.8483484), (120.96, 1.0, 1732284821.0560224), (120.92, 1.0, 1732284821.1714933), (121.05, 1.0, 1732284941.942514), (121.04, 1.0, 1732284982.2522135), (121.05, 1.0, 1732284983.5595126), (121.05, 1.0, 1732284983.5605583), (121.05, 1.0, 1732284983.9691958), (121.07, 1.0, 1732284983.9721465), (121.05, 1.0, 1732284984.6300185), (121.05, 1.0, 1732284985.5330987), (121.05, 1.0, 1732284986.2129629), (121.24, 1.0, 1732285018.4700956), (121.3, 1.0, 1732285130.0208497), (121.3, 1.0, 1732285210.8165665), (121.22, 1.0, 1732285216.5649831), (121.22, 1.0, 1732285216.5655468), (121.21, 1.0, 1732285217.0094173), (121.08, 1.0, 1732285235.9013577), (121.19, 1.0, 1732285422.3450158), (121.2, 1.0, 1732285425.5875647), (121.22, 1.0, 1732285425.5875647), (121.14, 1.0, 1732285538.8679624), (121.08, 1.0, 1732285543.7482438), (121.08, 4.0, 1732285543.7514477), (120.99, 1.0, 1732285721.1471565), (121.0, 2.0, 1732285762.3943186), (121.06, 1.0, 1732285795.0221956), (121.09, 1.0, 1732285795.2375436), (121.16, 1.0, 1732285826.2213693), (121.16, 1.0, 1732285918.5499184), (121.17, 2.0, 1732285934.1474392), (121.21, 1.0, 1732286028.1610093), (121.11, 1.0, 1732286074.010154), (121.22, 1.0, 1732286082.2049904), (121.25, 1.0, 1732286082.336928), (121.18, 1.0, 1732286083.9870968), (121.11, 1.0, 1732286084.1639102), (121.18, 1.0, 1732286100.3771293), (121.26, 1.0, 1732286100.3976555), (121.28, 1.0, 1732286128.0787635), (121.28, 1.0, 1732286130.708459), (121.33, 1.0, 1732286137.9329932), (121.25, 2.0, 1732286162.6380084), (121.25, 1.0, 1732286162.6493373), (121.25, 1.0, 1732286162.6494117), (121.3, 1.0, 1732286304.8124387), (121.3, 1.0, 1732286304.9064708), (121.23, 1.0, 1732286309.0100331), (121.29, 1.0, 1732286342.6977322), (121.26, 1.0, 1732286416.1831825), (121.25, 1.0, 1732286416.1838045), (121.2, 2.0, 1732286526.9183352), (121.19, 1.0, 1732286540.2252803), (121.2, 1.0, 1732286574.9975915), (121.13, 1.0, 1732286586.4669535), (121.13, 1.0, 1732286586.471102), (121.15, 1.0, 1732286586.7265842), (121.13, 1.0, 1732286587.5098572), (121.11, 1.0, 1732286588.06346), (121.0, 1.0, 1732286623.0101907)]



In [12]:
tr_class = TR_class(tau=10, tau_ema=10)

lag_trades

# Process the trade data to calculate tick imbalance indices.
indices = tr_class.tick_imbalance_single(lag_trades)

In [13]:
# Extract the Unix timestamps (first element of each tuple)
timestamps = [item[2] for item in indices]

# Convert the Unix timestamps to datetime (assuming nanosecond precision)
datetimes = converted_data = [datetime.fromtimestamp(ts).isoformat() for ts in timestamps]

# Recreate the list with converted datetime and the original second value
result = [(dt,p, v, item[3]) for dt, item,p, v in zip(datetimes, indices,[x[0] for x in lag_trades], [x[1] for x in lag_trades])]
result

[('2024-11-22T09:01:06.433828', 124.8, 1.0, None),
 ('2024-11-22T09:01:06.434665', 124.8, 1.0, None),
 ('2024-11-22T09:01:29.289333', 124.8, 1.0, None),
 ('2024-11-22T09:01:47.806319', 124.8, 1.0, None),
 ('2024-11-22T09:02:00.136390', 124.8, 1.0, None),
 ('2024-11-22T09:02:03.148714', 124.73, 1.0, None),
 ('2024-11-22T09:02:07.564892', 124.73, 1.0, None),
 ('2024-11-22T09:02:10.597795', 124.73, 1.0, None),
 ('2024-11-22T09:02:25.459382', 124.65, 1.0, None),
 ('2024-11-22T09:03:25.885635', 124.66, 1.0, 1),
 ('2024-11-22T09:03:51.339017', 124.2, 2.0, 2),
 ('2024-11-22T09:03:59.608357', 124.2, 3.0, 2),
 ('2024-11-22T09:03:59.608357', 124.19, 2.0, 2),
 ('2024-11-22T09:05:07.484876', 124.09, 3.0, 2),
 ('2024-11-22T09:05:07.484876', 124.09, 1.0, 2),
 ('2024-11-22T09:05:07.485064', 124.09, 1.0, 2),
 ('2024-11-22T09:05:19.679844', 124.07, 1.0, 3),
 ('2024-11-22T09:07:22.299234', 123.84, 1.0, 3),
 ('2024-11-22T09:07:24.760524', 123.96, 1.0, 3),
 ('2024-11-22T09:07:24.871637', 123.96, 1.0, 3),
